# LFM Tiling Example

This notebook demonstrates the configuration-driven lunar tiling API. It creates and visualizes two mixed-modality examples on the LTM grid:

1. A product-scoped WAC cube plus the canonical 63-band static cube.
2. A product-scoped NAC cube plus the same static context.

All raster sources use existing, explicitly configured vector indexes and bilinear resampling. Results are returned as `TileCubeRecord` objects, so downstream code uses structured source, zone, zoom, and tile fields instead of parsing output filenames. The legacy notebook remains at `notebooks/toy_model/tiling_example.ipynb` during migration.

## Imports and repository discovery

Run this notebook from the repository's top-level `notebooks/` directory. JupyterHub may expose the clone through `/panfs`; repository discovery normalizes that path to the equivalent `/explore` symlink before importing LFM.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import warnings

from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

In [ ]:
repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists() or not (repo_root / "model").exists():
    raise FileNotFoundError(
        "Cannot find the lfm/ and model/ directories. Run this notebook "
        "from the repository's top-level notebooks/ directory."
    )

sys.path.insert(0, str(repo_root))

from model import (
    TileConfig,
    TileSourceConfig,
    create_tiles_for_aoi,
    create_tiles_for_index,
    create_tiles_for_point,
)
from lfm.all_models.all_tasks.tiling_utils import (
    DEFAULT_NAC_BAND_NUMBER,
    DEFAULT_WAC_BAND_NUMBER,
    DEFAULT_ZOOM_LEVEL,
    RUN_ID,
    make_static_source,
    validate_path_pairs,
)
from lfm.all_models.all_tasks.viz import (
    pair_dynamic_and_static,
    plot_cube_pairs,
    print_record_summary,
)

print(f"Repository root: {repo_root}")
print("Successfully imported the modern LFM tiling API")

## User configuration

These are the values a notebook user is expected to edit for a normal tiling run. The defaults use representative Explore data exercised by the tiling validation suite.

### Data paths

- `PROJECT_DATA_DIR`: shared project-data root. The modality directories below are derived from it by default.
- `WAC_DATA_DIR`: directory containing processed WAC GeoTIFFs.
- `NAC_DATA_DIR`: directory containing processed NAC GeoTIFFs.
- `STATIC_DATA_DIR`: directory containing the static lunar raster links/files.

Each modality must have an existing `.shp` or `.gpkg` raster index. The default index paths are derived in the next section; tiling reads these indexes but never creates or refreshes them.

### Product selection and AOIs

- `WAC_PRODUCT_ID` and `NAC_PRODUCT_ID`: exact product prefixes to select from the corresponding raster index, such as the `M...` portion before the first period in a source filename.
- `WAC_AOI_BOUNDS` and `NAC_AOI_BOUNDS`: independent rectangular query bounds in lunar geographic IAU:30100 latitude/longitude. `ul_*` is the upper-left corner and `lr_*` is the lower-right corner. The selected product must intersect its AOI.

AOI queries discover intersecting LTM zones automatically. NAC coverage is sparse, so a NAC AOI may produce fewer dynamic/static pairs than its number of intersecting tiles.

### Output behavior

Generated cubes and plots are written beneath `repo_root/outputs/tiling/<RUN_ID>/`. The timestamped run directory prevents one execution from silently reusing another execution's cubes.

In [ ]:
PROJECT_DATA_DIR = Path("/explore/nobackup/projects/lfm")  # Shared project-data root on Explore.
WAC_DATA_DIR = PROJECT_DATA_DIR / "processed_data/Lunar/LRO_WAC_Pho_Sites"  # Processed WAC GeoTIFF directory.
NAC_DATA_DIR = PROJECT_DATA_DIR / "processed_data/Lunar/LRO_NAC_Pho_Sites"  # Processed NAC GeoTIFF directory.
STATIC_DATA_DIR = PROJECT_DATA_DIR / "staticLinks"  # Static lunar raster directory.

WAC_PRODUCT_ID = "M1107459759CE"  # Exact WAC product prefix to select.
NAC_PRODUCT_ID = "M1117899885LE"  # Exact NAC product prefix to select.

# IAU:30100 latitude/longitude bounds intersecting WAC_PRODUCT_ID.
WAC_AOI_BOUNDS = {
    "ul_lat": 31.241501479374616,  # Northern edge.
    "ul_lon": 121.01616204071722,  # Western edge.
    "lr_lat": 30.201163639632075,  # Southern edge.
    "lr_lon": 122.22643208573085,  # Eastern edge.
}

# IAU:30100 latitude/longitude bounds centered inside NAC_PRODUCT_ID.
NAC_AOI_BOUNDS = {
    "ul_lat": 1.0786543156953,  # Northern edge.
    "ul_lon": 149.752054273755,  # Western edge.
    "lr_lat": 1.0586543156953,  # Southern edge.
    "lr_lon": 149.772054273755,  # Eastern edge.
}

## Setting up variable values

The values in this section are derived paths, repository defaults, and notebook display controls. Most users can leave them unchanged.

### Raster indexes

- `WAC_INDEX`, `NAC_INDEX`, and `STATIC_INDEX`: existing vector indexes describing the geographic footprints and file locations of each source collection. Change one only if that collection uses a different index filename.
- `LOCATION_FIELD`: field in each vector index containing the raster path. Paths may be absolute or relative to the corresponding data directory.

### Tiling resolution

- `WAC_ZOOM_LEVEL`: LTM zoom used for WAC and its paired static cubes. Zoom 5 is the repository default and has approximately 75.8 m pixels.
- `NAC_ZOOM_LEVEL`: LTM zoom used for NAC and its paired static cubes. Zoom 11 has approximately 1.185 m pixels, close to the 1 m processed NAC source. Every source within one `TileConfig` shares its zoom and pixel grid.

### Outputs and visualization

- `BASE_OUTPUT_DIR`: parent directory for tiling notebook runs.
- `OUTPUT_DIR`: timestamped directory for this run's cubes and plots. It is created with `exist_ok=False` to prevent accidental reuse.
- `WAC_BAND_NUMBER` and `NAC_BAND_NUMBER`: 1-based dynamic-image band numbers to display; these do not limit which bands are written to a cube.
- `STATIC_BAND_TO_PLOT`: exact static cube band name to display; it does not limit the canonical 63 bands written to the static cube.
- `MAX_PLOT_TILES`: maximum number of dynamic/static tile pairs included in each figure. All intersecting cubes are still written.
- `RUN_ALTERNATE_QUERIES`: set to `True` to additionally demonstrate point and explicit tile-index queries after the AOI examples.

In [ ]:
WAC_INDEX = WAC_DATA_DIR / "output_index.shp"  # Geographic WAC footprint/file index.
NAC_INDEX = NAC_DATA_DIR / "output_index.shp"  # Geographic NAC footprint/file index.
STATIC_INDEX = STATIC_DATA_DIR / "db2.shp"  # Geographic static footprint/file index.
LOCATION_FIELD = "location"  # Index field containing each raster path.

WAC_ZOOM_LEVEL = DEFAULT_ZOOM_LEVEL  # Zoom 5: approximately 75.8 m/pixel.
NAC_ZOOM_LEVEL = 11  # 1.185 m/pixel; close to the processed NAC resolution

BASE_OUTPUT_DIR = repo_root / "outputs" / "tiling"  # Parent for all notebook tiling runs.
OUTPUT_DIR = BASE_OUTPUT_DIR / RUN_ID  # Unique timestamped directory for this run.
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)  # Refuse to reuse an existing run directory.

# Display choices do not change which cubes or bands are written.
WAC_BAND_NUMBER = DEFAULT_WAC_BAND_NUMBER  # First WAC VIS channel.
NAC_BAND_NUMBER = DEFAULT_NAC_BAND_NUMBER  # NAC single channel.
STATIC_BAND_TO_PLOT = "lola_kaguya_60mpp_elv"  # Exact static elevation band name.
MAX_PLOT_TILES = 4  # Maximum tile pairs per saved figure.
RUN_ALTERNATE_QUERIES = False  # True also runs point and tile-index examples.

print(f"Notebook outputs will default here: {OUTPUT_DIR}")

## Resolve data paths

This section fails early if a configured directory or index is unavailable, then constructs the canonical static source used by both examples. `STATIC_SOURCE` selects every intersecting static raster, preserves the repository's 63-band order, uses bilinear tiling, and converts declared source sentinels to the shared static output NoData value of `-32768`.

In [ ]:
# Confirm that every configured raster collection directory is available.
validate_path_pairs(
    {
        "WAC data directory": WAC_DATA_DIR,
        "NAC data directory": NAC_DATA_DIR,
        "static data directory": STATIC_DATA_DIR,
    },
    path_type="directory",
)

# Confirm that every read-only raster footprint index is available.
validate_path_pairs(
    {
        "WAC index": WAC_INDEX,
        "NAC index": NAC_INDEX,
        "static index": STATIC_INDEX,
    },
    path_type="file",
)

# Reuse one canonical static source definition at each example's chosen zoom.
STATIC_SOURCE = make_static_source(
    data_dir=STATIC_DATA_DIR,  # Directory containing the indexed static rasters.
    index_path=STATIC_INDEX,  # Existing static footprint/file index.
    location_field=LOCATION_FIELD,  # Index field containing raster paths.
)

## Example 1: WAC + STATIC AOI tiling

WAC uses `product_id` selection and preserves its native per-band NoData metadata. Static uses `all_intersecting`, the canonical band order, and standardized output NoData. Source order controls the order of records returned within each tile.

In [ ]:
wac_source = TileSourceConfig(
    name="wac",  # Stable source name used by selectors and result records.
    data_dir=WAC_DATA_DIR,  # Directory used to resolve indexed WAC paths.
    index_path=WAC_INDEX,  # Read-only geographic WAC footprint index.
    location_field=LOCATION_FIELD,  # Index field containing raster paths.
    selection_mode="product_id",  # Require one WAC product selector per query.
    resampling="bilinear",  # Required resampling contract for tiling.
    preserve_source_nodata=True,  # Keep native WAC NoData in its output cube.
)
wac_static_config = TileConfig(
    output_dir=OUTPUT_DIR / "wac_static",  # WAC/static cubes for this example.
    zoom_level=WAC_ZOOM_LEVEL,  # Shared output grid for both configured sources.
    sources=(wac_source, STATIC_SOURCE),  # Source order is preserved in results.
)
wac_static_config

In [ ]:
wac_static_records = create_tiles_for_aoi(
    wac_static_config,
    **WAC_AOI_BOUNDS,
    selectors={"wac": WAC_PRODUCT_ID},
)
print_record_summary(wac_static_records)

wac_static_pairs = pair_dynamic_and_static(wac_static_records, "wac")
print(f"WAC/static pairs available for plotting: {len(wac_static_pairs)}")

In [ ]:
wac_figure = plot_cube_pairs(
    wac_static_pairs,
    dynamic_label="WAC",
    dynamic_band_number=WAC_BAND_NUMBER,
    static_band_name=STATIC_BAND_TO_PLOT,
    output_path=OUTPUT_DIR / "plots" / "wac_static_cubes.png",
    max_tiles=MAX_PLOT_TILES,
)

## Example 2: NAC + STATIC AOI tiling

The same public API handles NAC without a WAC alias. This example uses LTM zoom 11 (approximately 1.185 m/pixel) to stay close to the processed NAC source's 1 m resolution and avoid placing its narrow footprint inside a much larger zoom-5 tile. NAC is declared `required=False` because the selected observation is sparse: an AOI tile without that NAC product is an expected skip, while static context is still produced for every intersecting tile. Plot pairing uses only tiles containing both modalities.

In [ ]:
nac_source = TileSourceConfig(
    name="nac",  # Stable source name used by selectors and result records.
    data_dir=NAC_DATA_DIR,  # Directory used to resolve indexed NAC paths.
    index_path=NAC_INDEX,  # Read-only geographic NAC footprint index.
    location_field=LOCATION_FIELD,  # Index field containing raster paths.
    selection_mode="product_id",  # Require one NAC product selector per query.
    resampling="bilinear",  # Required resampling contract for tiling.
    preserve_source_nodata=True,  # Keep native NAC NoData in its output cube.
    required=False,  # Skip tiles without this sparse NAC instead of failing.
)
nac_static_config = TileConfig(
    output_dir=OUTPUT_DIR / "nac_static",  # NAC/static cubes for this example.
    zoom_level=NAC_ZOOM_LEVEL,  # Shared near-native grid for NAC and static.
    sources=(nac_source, STATIC_SOURCE),  # Source order is preserved in results.
)
nac_static_config

In [ ]:
nac_static_records = create_tiles_for_aoi(
    nac_static_config,
    **NAC_AOI_BOUNDS,
    selectors={"nac": NAC_PRODUCT_ID},
)
print_record_summary(nac_static_records)

nac_static_pairs = pair_dynamic_and_static(nac_static_records, "nac")
print(f"NAC/static pairs available for plotting: {len(nac_static_pairs)}")

In [ ]:
nac_figure = plot_cube_pairs(
    nac_static_pairs,
    dynamic_label="NAC",
    dynamic_band_number=NAC_BAND_NUMBER,
    static_band_name=STATIC_BAND_TO_PLOT,
    output_path=OUTPUT_DIR / "plots" / "nac_static_cubes.png",
    max_tiles=MAX_PLOT_TILES,
)

## Alternative query entry points

The examples above use a geographic AOI. The same `TileConfig` contract supports a point query and an explicit LTM tile-index query. Set `RUN_ALTERNATE_QUERIES = True` in the user configuration to run the examples below. Separate output directories prevent these calls from overwriting the AOI outputs.

In [ ]:
if RUN_ALTERNATE_QUERIES:
    example_wac_record = next(
        record for record in wac_static_records if record.source_name == "wac"
    )
    example_lat = (WAC_AOI_BOUNDS["ul_lat"] + WAC_AOI_BOUNDS["lr_lat"]) / 2
    example_lon = (WAC_AOI_BOUNDS["ul_lon"] + WAC_AOI_BOUNDS["lr_lon"]) / 2

    point_config = TileConfig(
        output_dir=OUTPUT_DIR / "wac_static_point",
        zoom_level=WAC_ZOOM_LEVEL,
        sources=wac_static_config.sources,
    )
    point_records = create_tiles_for_point(
        point_config,
        lat=example_lat,
        lon=example_lon,
        zone=example_wac_record.zone,
        selectors={"wac": WAC_PRODUCT_ID},
    )
    print("Point-query records:")
    print_record_summary(point_records)

    index_config = TileConfig(
        output_dir=OUTPUT_DIR / "wac_static_tile_index",
        zoom_level=WAC_ZOOM_LEVEL,
        sources=wac_static_config.sources,
    )
    index_records = create_tiles_for_index(
        index_config,
        tile_x=example_wac_record.tile_x,
        tile_y=example_wac_record.tile_y,
        zone=example_wac_record.zone,
        selectors={"wac": WAC_PRODUCT_ID},
    )
    print("Tile-index-query records:")
    print_record_summary(index_records)
else:
    print("Alternate point and tile-index queries are configured but disabled.")

## Outputs

The run directory contains separate WAC/static and NAC/static cube directories plus saved PNG visualizations. Each returned record contains the authoritative modality and tile identity; filenames remain descriptive for human inspection but are not the machine-readable interface.

In [ ]:
print(f"Completed tiling notebook run: {OUTPUT_DIR}")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT_DIR)}")